<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-08-agents-and-adk/lesson-8.4-a2a/notebooks/GCP_Capstone_8.4_A2A.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8.4 A2A Protocol: Agent Cards, JSON-RPC, Cross-Agent Tasks
**Netsetos GenAI Engineering — GCP Capstone**


In [ ]:
!pip install -q 'google-adk[a2a]'
import os
os.environ['GOOGLE_API_KEY'] = 'YOUR-KEY'
print('ADK + A2A ready')


## Cell 1: Create DocuMind Agent


In [ ]:
from google.adk.agents import LlmAgent
from google.adk.tools import ToolContext

def search_documents(query: str, tool_context: ToolContext) -> dict:
    """Search documents.
    Args:
        query: Search query.
    """
    return {'results': [{'id': 'D-01', 'title': 'Q1 Report'}]}

def summarize_document(document_id: str, summary_type: str) -> dict:
    """Summarize a document.
    Args:
        document_id: Document ID.
        summary_type: brief/detailed/executive.
    """
    return {'summary': f'Summary of {document_id}'}

root_agent = LlmAgent(
    name='documind',
    model='gemini-2.5-flash',
    description='Document analysis, summarization, and Q&A agent',
    instruction='You are DocuMind AI. Search and summarize documents.',
    tools=[search_documents, summarize_document],
)
print(f'Agent: {root_agent.name}')


## Cell 2: Wrap as A2A Server


In [ ]:
from google.adk.a2a.utils.agent_to_a2a import to_a2a

# ONE LINE: convert ADK agent to A2A server
a2a_app = to_a2a(root_agent, port=8001)
print('A2A server created')
print('Run: uvicorn agent:a2a_app --host 0.0.0.0 --port 8001')
print('Agent Card: http://localhost:8001/.well-known/agent.json')


## Cell 3: Write Agent Card (Custom)


In [ ]:
import json

agent_card = {
    'name': 'DocuMind Intelligence Agent',
    'description': 'Document analysis, summarization, and Q&A',
    'url': 'https://documind-a2a.run.app/',
    'version': '1.0.0',
    'defaultInputModes': ['text/plain', 'application/pdf'],
    'defaultOutputModes': ['text/plain', 'application/json'],
    'capabilities': {'streaming': True, 'pushNotifications': True},
    'skills': [
        {
            'id': 'document-summarization',
            'name': 'Document Summarization',
            'description': 'Summarize uploaded documents with configurable detail',
            'tags': ['summarization', 'documents'],
            'examples': ['Summarize this quarterly report']
        },
        {
            'id': 'document-qa',
            'name': 'Document Q&A',
            'description': 'Answer questions about documents using RAG',
            'tags': ['qa', 'rag', 'documents'],
            'examples': ['What revenue was reported in Q3?']
        }
    ]
}

with open('agent-card.json', 'w') as f:
    json.dump(agent_card, f, indent=2)
print(json.dumps(agent_card, indent=2))


## Cell 4: A2A Client — Connect to Remote Agent


In [ ]:
from google.adk.agents import Agent
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent

# Connect to a remote A2A agent (e.g., Currency Agent)
# Uncomment when a remote agent is running:
# currency_agent = RemoteA2aAgent(
#     name='currency_agent',
#     description='Converts currencies using live exchange rates',
#     agent_card='http://localhost:10000/.well-known/agent.json',
# )
#
# root_with_remote = Agent(
#     model='gemini-2.5-flash',
#     name='documind_connected',
#     instruction='Delegate currency questions to currency_agent.',
#     sub_agents=[currency_agent],
#     tools=[search_documents, summarize_document],
# )
print('RemoteA2aAgent wraps any A2A server as an ADK sub-agent')
print('The LLM sees it as a regular sub-agent (transfer_to_agent)')


## Cell 5: A2A vs MCP Comparison


In [ ]:
comparison = {
    'MCP (Module 7)': {
        'connects': 'Agent -> Tool/Data',
        'analogy': 'USB port',
        'transport': 'JSON-RPC (stdio/HTTP)',
        'state': 'Stateless tool calls',
        'use_for': 'Databases, APIs, file systems',
        'example': 'BigQuery via MCP Toolbox'
    },
    'A2A (Lesson 8.4)': {
        'connects': 'Agent <-> Agent',
        'analogy': 'Internet',
        'transport': 'JSON-RPC (HTTP/gRPC)',
        'state': 'Task lifecycle (9 states)',
        'use_for': 'Autonomous peers with reasoning',
        'example': 'Translation Agent, Compliance Agent'
    }
}

for protocol, details in comparison.items():
    print(f'\n{protocol}:')
    for k, v in details.items():
        print(f'  {k}: {v}')

print('\nUse BOTH together:')
print('  MCP for vertical (tools) + A2A for horizontal (agents)')


## Cell 6: JSON-RPC Request Example


In [ ]:
import json

# A2A message/send request
request = {
    'jsonrpc': '2.0',
    'id': 1,
    'method': 'message/send',
    'params': {
        'message': {
            'role': 'user',
            'parts': [{
                'kind': 'text',
                'text': 'Summarize this contract and flag compliance risks'
            }]
        }
    }
}
print('A2A JSON-RPC request:')
print(json.dumps(request, indent=2))

# Task lifecycle states
states = {
    'in_progress': ['submitted', 'working'],
    'interrupted': ['input-required', 'auth-required'],
    'terminal': ['completed', 'failed', 'canceled', 'rejected']
}
print('\nTask lifecycle states:')
for category, s in states.items():
    print(f'  {category}: {s}')


## Done! MODULE 8 COMPLETE!

**Full Module 8 journey:**
- 8.1: Root Agent (LlmAgent, tools, instruction design)
- 8.2: Multi-Agent Orchestration (Sequential, Parallel, Loop)
- 8.3: Agent Engine (Memory Bank, context, HIPAA)
- 8.4: A2A Protocol (Agent Cards, JSON-RPC, cross-agent tasks)

**DocuMind is now a production-grade, memory-enabled, multi-vendor agent system.**
